### 1. IMPORTS

In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from webdriver_manager.chrome import ChromeDriverManager

from bs4 import BeautifulSoup

import pandas as pd
import time

### 2. LAUNCH BROWSER

In [2]:
url = "https://www.fifa.com/en/tournaments/mens/worldcup/canadamexicousa2026/statistics/player-statistics"

driver = webdriver.Chrome(
    service=Service(
        ChromeDriverManager().install()
    )
)

driver.maximize_window()
driver.get(url)

time.sleep(5)
print(driver.title)

Player Stats | FIFA World Cup 2026™


### 3. FIFA CATEGORIES

In [3]:
categories = [
    "General",
    "Attacking",
    "Distribution",
    "Defending",
    "Discipline",
    "Goalkeeping",
    "Movement",
    "Physical"
]

### 4. CLICK CATEGORY FUNCTION

In [4]:
def click_category(category):
    if category == "General":
        return

    button = WebDriverWait(
        driver,
        15
    ).until(
        EC.element_to_be_clickable(
            (
                By.XPATH,
                f"//button[normalize-space()='{category}']"
            )
        )
    )

    driver.execute_script(
        "arguments[0].click();",
        button
    )

    time.sleep(5)
    print(category, "opened")

### 5. TABLE EXTRACTION FUNCTION

In [5]:
def extract_table():
    soup = BeautifulSoup(
        driver.page_source,
        "html.parser"
    )

    table = soup.find("table")

    rows = table.find_all("tr")

    headers = []

    for th in rows[0].find_all("th"):

        headers.append(
            th.get_text(strip=True)
        )

    data = []

    for row in rows[1:]:

        cols = row.find_all("td")

        if len(cols) > 0:
            row_data = []

            for index, col in enumerate(cols):

                if index == 1:

                    player_name = col.find(
                        "div",
                        class_="main-text"
                    )

                    if player_name:

                        row_data.append(
                            player_name.get_text(strip=True)
                        )

                    else:

                        row_data.append(
                            col.get_text(strip=True)
                        )

                else:

                    row_data.append(
                        col.get_text(strip=True)
                    )

            data.append(row_data)

    df = pd.DataFrame(
        data,
        columns=headers
    )

    return df

### 6. TEST CATEGORY CHANGE

In [6]:
click_category("Distribution")

test_df = extract_table()

print(test_df.columns)
test_df.head()

Distribution opened
Index(['Rank', 'Player', 'Passes', 'Passes Completed', 'Passing Accuracy (%)',
       'Crosses', 'Crossing Accuracy (%)', 'Take-Ons Completed',
       'Defensive Linebreaks Attempted', 'Defensive Linebreaks Acc (%)',
       'Switches of Play Attempted', 'Switches of Play Acc (%)'],
      dtype='str')


,Rank,Player,Passes,Passes Completed,Passing Accuracy (%),Crosses,Crossing Accuracy (%),Take-Ons Completed,Defensive Linebreaks Attempted,Defensive Linebreaks Acc (%),Switches of Play Attempted,Switches of Play Acc (%)
0,1,Rodri,799,747,93,3,0,2,21,90,11,91
1,2,Pau Cubarsi,690,668,97,1,0,0,7,71,2,100
2,3,Aymeric Laporte,660,618,94,0,0,0,12,58,8,100
3,4,Leandro Paredes,565,532,94,2,50,3,8,38,7,100
4,5,Marc Guehi,532,514,97,0,0,2,10,70,4,75


### 7. SCRAPE ALL CATEGORIES

In [7]:
all_tables = {}

for category in categories:
    print("\nExtracting:", category)

    click_category(category)

    df = extract_table()

    all_tables[category] = df

    print(
        category,
        df.shape
    )


Extracting: General
General (50, 12)

Extracting: Attacking
Attacking opened
Attacking (50, 12)

Extracting: Distribution
Distribution opened
Distribution (50, 12)

Extracting: Defending
Defending opened
Defending (50, 6)

Extracting: Discipline
Discipline opened
Discipline (50, 8)

Extracting: Goalkeeping
Goalkeeping opened
Goalkeeping (50, 5)

Extracting: Movement
Movement opened
Movement (50, 12)

Extracting: Physical
Physical opened
Physical (50, 6)


### .8 VERIFY TABLES

In [8]:
for category, df in all_tables.items():

    print("\n")
    print(category)
    print(
        list(df.columns)
    )



General
['Rank', 'Player', 'Passes', 'Passes Completed', 'Passing Accuracy (%)', 'Crosses', 'Crossing Accuracy (%)', 'Take-Ons Completed', 'Defensive Linebreaks Attempted', 'Defensive Linebreaks Acc (%)', 'Switches of Play Attempted', 'Switches of Play Acc (%)']


Attacking
['Rank', 'Player', 'Assists', 'Attempts On Target', 'Attempts At Goal', 'Attempts At Goal Conv. Rate (%)', 'Attempts Inside the Penalty Area', 'Attempts Outside the Penalty Area', 'Headed Attempts at Goal', 'xG', 'xG Efficiency', 'Corners']


Distribution
['Rank', 'Player', 'Passes', 'Passes Completed', 'Passing Accuracy (%)', 'Crosses', 'Crossing Accuracy (%)', 'Take-Ons Completed', 'Defensive Linebreaks Attempted', 'Defensive Linebreaks Acc (%)', 'Switches of Play Attempted', 'Switches of Play Acc (%)']


Defending
['Rank', 'Player', 'Own Goals', 'Forced Turnovers', 'Defensive Pressures Applied', 'Defensive Pressures Directly Applied']


Discipline
['Rank', 'Player', 'Fouls Against', 'Fouls For', 'Yellow Cards',

### 9. SAVE CATEGORY FILES

In [9]:
for category, df in all_tables.items():
    filename = (
        "../data/"
        + category.lower()
        + ".csv"
    )

    df.to_csv(
        filename,
        index=False
    )

print("Category files saved")

Category files saved


### 10. CREATE FINAL RAW DATASET

In [10]:
combined_df = pd.concat(
    all_tables.values(),
    keys=all_tables.keys()
)

combined_df = combined_df.reset_index()

combined_df.rename(
    columns={
        "level_0":"Category"
    },
    inplace=True
)

combined_df.head()

,Category,level_1,Rank,Player,Passes,Passes Completed,Passing Accuracy (%),Crosses,Crossing Accuracy (%),Take-Ons Completed,...,Offers Inside Team Shape,Offers Outside Team Shape,Receptions In Behind,Receptions Between Midfield And Defensive Line,Receptions Under Pressure,Player Involvements,Top Speed (km/h),High Speed Running,Sprints,Total Distance (m)
0,General,0,1,Rodri,799,747,93,3,0,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,General,1,2,Pau Cubarsi,690,668,97,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,General,2,3,Aymeric Laporte,660,618,94,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,General,3,4,Leandro Paredes,565,532,94,2,50,3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,General,4,5,Marc Guehi,532,514,97,0,0,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 11. EXPORT PLAYERS_RAW.CSV

In [11]:
combined_df.to_csv(
    "../data/players_raw.csv",
    index=False
)

print(
    "players_raw.csv created successfully!"
)

players_raw.csv created successfully!


### 12. FINAL CHECK

In [12]:
print(
    combined_df.shape
)

combined_df.head()

(400, 51)


,Category,level_1,Rank,Player,Passes,Passes Completed,Passing Accuracy (%),Crosses,Crossing Accuracy (%),Take-Ons Completed,...,Offers Inside Team Shape,Offers Outside Team Shape,Receptions In Behind,Receptions Between Midfield And Defensive Line,Receptions Under Pressure,Player Involvements,Top Speed (km/h),High Speed Running,Sprints,Total Distance (m)
0,General,0,1,Rodri,799,747,93,3,0,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,General,1,2,Pau Cubarsi,690,668,97,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,General,2,3,Aymeric Laporte,660,618,94,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,General,3,4,Leandro Paredes,565,532,94,2,50,3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,General,4,5,Marc Guehi,532,514,97,0,0,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
